# Tuần 1: truy xuất và phân đoạn trên dữ liệu thật

Notebook sinh tự động bởi `scripts/build_colab_notebook.py` từ `src/ragvi/`; đừng sửa tay, hãy sửa mã nguồn rồi sinh lại (sinh lại sẽ xóa output). Chạy từ trên xuống trên Colab runtime, thứ tự cell theo thứ tự ưu tiên của kế hoạch ngày 13–18/11.

### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Cài thư viện

Runtime mới mất các package cài thêm. Mong đợi cài xong không lỗi; phục vụ mọi ngày 13–18/11.

In [1]:
%pip install -q rank_bm25 underthesea sentence-transformers datasets

### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Kết nối Google Drive

Dữ liệu và kết quả nằm trên Drive. Mong đợi thấy `Mounted at /content/drive`.

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### [TÁI SỬ DỤNG] Đường dẫn

`P` là dữ liệu đã làm sạch, `O` là thư mục kết quả Tuần 1, `EVAL` chứa tập đánh giá đóng băng. Cell cũng tạo `/content/src/ragvi/` để các cell sau ghi mã vào.

In [3]:
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/Vietnamese-RAG-Research')
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Không thấy {PROJECT_DIR}. Hãy chạy cell kết nối Drive trước.')
P = PROJECT_DIR / 'data' / 'processed'
Z = P / 'zalo_legal'
O = PROJECT_DIR / 'results' / 'tuan1'
EVAL = PROJECT_DIR / 'data' / 'eval'
Path('/content/src/ragvi').mkdir(parents=True, exist_ok=True)
print(P, O, EVAL, sep='\n')

/content/drive/MyDrive/Vietnamese-RAG-Research/data/processed
/content/drive/MyDrive/Vietnamese-RAG-Research/results/tuan1
/content/drive/MyDrive/Vietnamese-RAG-Research/data/eval


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/__init__.py`

Runtime Colab không thấy `src/` ở máy cục bộ, nên mã được chép nguyên văn vào `/content/src/`. Mong đợi `Writing /content/src/ragvi/...`.

In [4]:
%%writefile /content/src/ragvi/__init__.py


Writing /content/src/ragvi/__init__.py


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/chunking.py`

Runtime Colab không thấy `src/` ở máy cục bộ, nên mã được chép nguyên văn vào `/content/src/`. Mong đợi `Writing /content/src/ragvi/...`.

In [5]:
%%writefile /content/src/ragvi/chunking.py
"""Phân đoạn: C1 (cố định), C3 (theo câu), C4 (nguyên tài liệu), C5 ViStructChunk (Điều → Khoản → Điểm).

Mọi hàm trả về danh sách (chunk_id, doc_id, start, end, text); start/end là vị trí ký tự trong
tài liệu gốc và text là lát cắt nguyên văn text[start:end] (riêng C5 có thể gắn thêm tiêu đề cha
ở phía trước). Token được đếm bằng một tokenizer tham chiếu truyền từ ngoài vào (D7: BAAI/bge-m3).

Chạy: PYTHONPATH=src python -m ragvi.chunking <processed>/zalo_legal --methods c1,c3,c4,c5,c5_nohead
"""
import argparse
import math
import re
import statistics
from bisect import bisect_left
from pathlib import Path

SIZE = 256
METHODS = ('c1', 'c3', 'c4', 'c5', 'c5_nohead')
CLAUSE = re.compile(r'^(\d+)\.\s', re.M)      # Khoản: "1. ..."
POINT = re.compile(r'^([a-zđ])\)\s', re.M)    # Điểm: "a) ..."


def _offsets(text, tok):
    """Trả về (offset_mapping, danh sách vị trí bắt đầu của từng token)."""
    off = tok(text, add_special_tokens=False, return_offsets_mapping=True)['offset_mapping']
    return off, [o[0] for o in off]


def _ntok(starts, s, e):
    """Số token bắt đầu trong khoảng ký tự [s, e)."""
    return bisect_left(starts, e) - bisect_left(starts, s)


def _fixed_spans(off, starts, s, e, size):
    """Cắt khoảng [s, e) thành các đoạn `size` token, không chồng lấn.

    XLM-R (bge-m3) có token "▁" đứng riêng mang offset trùng ký tự đầu của token sau (ví dụ "ỦY", "QĐ", "≥"),
    nên đầu cuối của mỗi đoạn bị chặn tại điểm bắt đầu của đoạn kế tiếp.
    """
    i0, i1 = bisect_left(starts, s), bisect_left(starts, e)
    return [(max(off[i][0], s), min(off[min(i + size, i1) - 1][1], off[i + size][0] if i + size < i1 else e, e))
            for i in range(i0, i1, size)]


def _sentence_spans(text, s, e, sent_split):
    """Vị trí (start, end) của từng câu trong text[s:e]; phần chữ hàm tách câu bỏ sót được giữ lại."""
    spans, pos = [], s
    for sent in sent_split(text[s:e]):
        sent = sent.strip()
        if not sent:
            continue
        a = text.find(sent, pos, e)
        b = a + len(sent)
        if a < 0:  # hàm tách câu đã đổi khoảng trắng: so khớp bỏ qua khoảng trắng
            m = re.compile(r'\s*'.join(map(re.escape, sent.split()))).search(text, pos, e)
            if not m:
                raise ValueError(f'Không tìm thấy câu trong văn bản gốc: {sent[:80]!r}')
            a, b = m.span()
        a -= len(text[pos:a].lstrip())  # chữ nằm giữa hai câu thuộc về câu sau
        spans.append((a, b))
        pos = b
    tail = text[pos:e]
    if tail.strip():
        spans.append((e - len(tail.lstrip()), pos + len(tail.rstrip())))
    return spans


def _sentence_merge_spans(text, off, starts, s, e, size, sent_split):
    """C3 trên khoảng [s, e): gộp câu tới `size` token; câu dài hơn `size` thì cắt bằng C1."""
    out, cur = [], None
    for a, b in _sentence_spans(text, s, e, sent_split):
        if cur and _ntok(starts, cur[0], b) <= size:
            cur = (cur[0], b)
            continue
        if cur:
            out.append(cur)
            cur = None
        if _ntok(starts, a, b) > size:
            out += _fixed_spans(off, starts, a, b, size)
        else:
            cur = (a, b)
    return out + [cur] if cur else out


def _default_sent_split():
    from underthesea import sent_tokenize
    return sent_tokenize


def _records(doc_id, text, spans):
    return [(f'{doc_id}#{i}', doc_id, s, e, text[s:e]) for i, (s, e) in enumerate(spans)]


def chunk_fixed(doc_id, text, tok, size=SIZE):
    """C1: mỗi chunk `size` token, không chồng lấn, cắt theo vị trí ký tự do tokenizer trả về."""
    off, starts = _offsets(text, tok)
    return _records(doc_id, text, _fixed_spans(off, starts, 0, len(text), size))


def chunk_sentence(doc_id, text, tok, size=SIZE, sent_split=None):
    """C3: tách câu (mặc định `underthesea.sent_tokenize`), gộp câu tới `size` token."""
    off, starts = _offsets(text, tok)
    spans = _sentence_merge_spans(text, off, starts, 0, len(text), size, sent_split or _default_sent_split())
    return _records(doc_id, text, spans)


def chunk_document(doc_id, text):
    """C4: nguyên tài liệu (đoạn ViQuAD / nguyên điều luật Zalo) là một chunk."""
    return [(f'{doc_id}#0', doc_id, 0, len(text), text)]


def has_clause(body):
    """Điều luật có ít nhất một khoản được nhận diện hay không."""
    return bool(CLAUSE.search(body))


def chunk_struct(doc_id, title, body, tok, size=SIZE, heading=True, sent_split=None):
    """C5 ViStructChunk cho một điều luật; `heading=False` là biến thể không tiêu đề (cùng ranh giới).

    Tài liệu gốc là `title + "\\n" + body` (giống `ragvi.data.load_zalo_corpus`), start/end tính trên
    chuỗi đó. Đơn vị chunk là Khoản; Khoản dài hơn `size` token thì tách theo Điểm; phần mở đầu trước
    khoản 1 (kèm dòng tiêu đề Điều) là một chunk riêng; điều luật không có khoản là một chunk, trùng C4.
    Mảnh nào vẫn dài hơn `size` token thì cắt bằng C3.
    Tiêu đề cha: "Văn bản 01/2009/TT-BNN, Điều 1. ..., Khoản 2[, Điểm a]: " + nội dung. Chunk bắt đầu
    từ đầu tài liệu đã chứa dòng tiêu đề Điều nên chỉ gắn "Văn bản ..., ".
    """
    text = title + '\n' + body
    off, starts = _offsets(text, tok)
    doc = 'Văn bản ' + doc_id.rsplit('+', 1)[0].upper()
    full = ', '.join(filter(None, [doc, title.strip()]))

    def cut(s, e, head):
        e = s + len(text[s:e].rstrip())
        if s >= e:
            return []
        if _ntok(starts, s, e) <= size:
            spans = [(s, e)]
        else:
            spans = _sentence_merge_spans(text, off, starts, s, e, size, sent_split or _default_sent_split())
        return [(a, b, f'{doc}, ' if a == 0 else f'{head}: ') for a, b in spans]

    # ponytail: nhận diện khoản chỉ bằng regex đầu dòng, không kiểm tra số khoản tăng dần;
    # nếu đối chiếu tay 30 điều (Ngày 7) thấy nhận nhầm danh sách đánh số thì thêm kiểm tra đó.
    clauses = list(CLAUSE.finditer(text, len(title) + 1))
    if not clauses:
        pieces = [(0, len(text), f'{doc}, ')] if len(off) <= size else cut(0, len(text), full)
    else:
        # Luôn giữ phần trước khoản 1, kể cả khi chỉ còn dòng tiêu đề Điều; bỏ đi thì c5_nohead mất tiêu đề.
        pieces = cut(0, clauses[0].start(), full)
        for m, nxt in zip(clauses, clauses[1:] + [None]):
            s, e = m.start(), nxt.start() if nxt else len(text)
            head = f'{full}, Khoản {m[1]}'
            points = list(POINT.finditer(text, s, e)) if _ntok(starts, s, e) > size else []
            if not points:
                pieces += cut(s, e, head)
                continue
            pieces += cut(s, points[0].start(), head)
            for p, q in zip(points, points[1:] + [None]):
                pieces += cut(p.start(), q.start() if q else e, f'{head}, Điểm {p[1]}')
    return [(f'{doc_id}#{i}', doc_id, s, e, (prefix if heading else '') + text[s:e])
            for i, (s, e, prefix) in enumerate(pieces)]


def chunk_stats(chunks, tok):
    """Số chunk và phân phối độ dài theo token (tính trên `text` của chunk, kể cả tiêu đề cha)."""
    # ponytail: token hóa lại từng chunk; nếu chậm trên toàn kho thì token hóa theo lô.
    lengths = sorted(len(_offsets(c[4], tok)[0]) for c in chunks)
    if not lengths:
        return {'n_chunks': 0, 'tok_mean': 0, 'tok_median': 0, 'tok_p95': 0, 'tok_max': 0}
    return {'n_chunks': len(lengths), 'tok_mean': statistics.fmean(lengths),
            'tok_median': statistics.median(lengths),
            'tok_p95': lengths[math.ceil(0.95 * len(lengths)) - 1], 'tok_max': lengths[-1]}


def clause_rate(bodies):
    """Tỷ lệ điều luật có ít nhất một khoản được nhận diện."""
    bodies = list(bodies)
    return sum(map(has_clause, bodies)) / len(bodies) if bodies else 0.0


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument('processed_dir', help='thư mục <processed>/zalo_legal')
    parser.add_argument('--methods', default=','.join(METHODS))
    parser.add_argument('--tokenizer', default='BAAI/bge-m3')
    parser.add_argument('--size', type=int, default=SIZE)
    args = parser.parse_args()
    methods = args.methods.split(',')
    if set(methods) - set(METHODS):
        parser.error(f'Cách phân đoạn không hợp lệ: {sorted(set(methods) - set(METHODS))}; chọn trong {METHODS}')

    from datasets import load_from_disk
    from transformers import AutoTokenizer

    from ragvi.data import load_zalo_corpus

    tok = AutoTokenizer.from_pretrained(args.tokenizer)
    docs = load_zalo_corpus(args.processed_dir)
    # C5 cần tiêu đề riêng, dict {doc_id: title + "\n" + text} không tách được chắc chắn nên đọc thẳng cột title.
    corpus = load_from_disk(str(Path(args.processed_dir) / 'corpus')).select_columns(['id', 'title'])
    titles = {row['id']: row['title'] for row in corpus}
    bodies = {}
    for doc_id, text in docs.items():
        if not text.startswith(titles[doc_id] + '\n'):
            raise ValueError(f'Tài liệu {doc_id} không có dạng title + "\\n" + text')
        bodies[doc_id] = text[len(titles[doc_id]) + 1:]

    run = {
        'c1': lambda d: chunk_fixed(d, docs[d], tok, args.size),
        'c3': lambda d: chunk_sentence(d, docs[d], tok, args.size),
        'c4': lambda d: chunk_document(d, docs[d]),
        'c5': lambda d: chunk_struct(d, titles[d], bodies[d], tok, args.size),
        'c5_nohead': lambda d: chunk_struct(d, titles[d], bodies[d], tok, args.size, heading=False),
    }
    print(f'Số tài liệu: {len(docs)}')
    print(f'{"cách":<10}{"số chunk":>10}{"tb":>9}{"trung vị":>10}{"p95":>8}{"max":>8}')
    for name in methods:
        st = chunk_stats([c for d in docs for c in run[name](d)], tok)
        print(f'{name:<10}{st["n_chunks"]:>10}{st["tok_mean"]:>9.1f}{st["tok_median"]:>10.1f}'
              f'{st["tok_p95"]:>8}{st["tok_max"]:>8}')
    print(f'Tỷ lệ điều luật có ít nhất một khoản: {clause_rate(bodies.values()):.1%}')


Writing /content/src/ragvi/chunking.py


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/data.py`

Runtime Colab không thấy `src/` ở máy cục bộ, nên mã được chép nguyên văn vào `/content/src/`. Mong đợi `Writing /content/src/ragvi/...`.

In [6]:
%%writefile /content/src/ragvi/data.py
"""Dựng tập đánh giá Zalo Legal và ViQuAD theo quyết định D1 (reports/quyet-dinh-thiet-ke.md).

Chạy trên Colab, từ thư mục gốc của kho mã:
    PYTHONPATH=src python -m ragvi.data zalo <processed>/zalo_legal <thư mục đầu ra> --seed 0
    PYTHONPATH=src python -m ragvi.data viquad <processed>/viquad <thư mục đầu ra> --seed 0
"""
import argparse
import hashlib
import json
import random
import re
from pathlib import Path

# Thứ tự là thứ tự ưu tiên khi xếp nhóm vào từng phần.
SIZES = {'unanswerable': 120, 'dev': 100, 'answerable': 280}


def load_zalo_queries(processed_dir):
    """Trả về (questions: {qid: text}, gold: {qid: set(doc_id)}) cho các truy vấn có nhãn."""
    from datasets import load_from_disk

    root = Path(processed_dir)
    gold = {}
    for row in load_from_disk(str(root / 'qrels')):
        if row['score'] > 0:
            gold.setdefault(row['query-id'], set()).add(row['corpus-id'])

    questions = {}
    for row in load_from_disk(str(root / 'queries')):
        qid, text = row['id'], row['text']
        if qid not in gold:
            continue
        if questions.setdefault(qid, text) != text:
            raise ValueError(f'Truy vấn trùng mã nhưng khác nội dung: {qid}')
    return questions, gold


def load_zalo_corpus(processed_dir, exclude=()):
    """Trả về {doc_id: title + "\\n" + text}; điều luật có text rỗng vẫn giữ (chỉ còn tiêu đề).

    exclude: các doc_id bị loại, ví dụ nội dung removed_doc_ids.json, để dựng kho thu hẹp.
    """
    from datasets import load_from_disk

    exclude = set(exclude)
    return {row['id']: row['title'] + '\n' + row['text']
            for row in load_from_disk(str(Path(processed_dir) / 'corpus')) if row['id'] not in exclude}


def gold_groups(gold):
    """Nhóm các truy vấn dùng chung điều luật vàng, trực tiếp hoặc bắc cầu (union-find)."""
    parent = {}

    def find(x):
        parent.setdefault(x, x)
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for qid, docs in gold.items():
        for doc in docs:
            parent[find(('q', qid))] = find(('d', doc))

    groups = {}
    for qid in gold:
        groups.setdefault(find(('q', qid)), []).append(qid)
    return sorted((sorted(g) for g in groups.values()), key=lambda g: g[0])


def split_zalo(gold, sizes=SIZES, seed=0):
    """Xếp trọn từng nhóm vào phần đầu tiên còn đủ chỗ; nhóm không vừa phần nào thì để dự trữ.

    Trả về (parts: {tên phần: [qid]}, group_of: {qid: chỉ số nhóm}).
    """
    groups = gold_groups(gold)
    random.Random(seed).shuffle(groups)
    parts = {name: [] for name in (*sizes, 'reserve')}
    group_of = {}
    for index, group in enumerate(groups):
        group_of.update((qid, index) for qid in group)
        name = next((n for n in sizes if len(parts[n]) + len(group) <= sizes[n]), 'reserve')
        parts[name] += group
    return parts, group_of


def build_zalo_eval(questions, gold, out_dir, sizes=SIZES, seed=0, tolerance=5):
    """Ghi test.jsonl, dev.jsonl, reserve.jsonl, removed_doc_ids.json, manifest.json.

    Từ chối ghi đè: tập đánh giá đã đóng băng thì không dựng lại vào cùng thư mục.
    """
    out = Path(out_dir)
    if out.exists() and any(out.iterdir()):
        raise FileExistsError(f'{out} đã có dữ liệu. Tập đánh giá đã đóng băng; hãy chọn thư mục khác.')

    parts, group_of = split_zalo(gold, sizes, seed)
    for name, target in sizes.items():
        if target - len(parts[name]) > tolerance:
            raise ValueError(f'Phần {name} chỉ có {len(parts[name])}/{target} câu; thử seed khác.')

    removed = sorted(set().union(*(gold[q] for q in parts['unanswerable'])))
    kept_gold = set().union(*(gold[q] for name in ('dev', 'answerable', 'reserve') for q in parts[name]))
    assert not kept_gold & set(removed), 'Điều luật bị loại vẫn là điều luật vàng của câu ở phần khác'

    def rows(names):
        return [{'qid': q, 'question': questions[q], 'answerable': name != 'unanswerable',
                 'gold_doc_ids': sorted(gold[q]), 'group': group_of[q]}
                for name in names for q in parts[name]]

    out.mkdir(parents=True, exist_ok=True)
    files = {'test.jsonl': rows(['answerable', 'unanswerable']), 'dev.jsonl': rows(['dev']),
             'reserve.jsonl': rows(['reserve'])}
    for name, data in files.items():
        (out / name).write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in data), encoding='utf-8')
    (out / 'removed_doc_ids.json').write_text(json.dumps(removed, ensure_ascii=False, indent=0), encoding='utf-8')

    manifest = {
        'dataset': 'zalo_legal', 'seed': seed, 'target_sizes': sizes,
        'actual_sizes': {name: len(qids) for name, qids in parts.items()},
        'labeled_queries': len(gold), 'groups': len(set(group_of.values())),
        'removed_docs': len(removed),
        'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(out.iterdir())},
    }
    (out / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest


def load_viquad(processed_dir):
    """Trả về danh sách dòng của cả ba split, mỗi dòng thêm khóa 'split'."""
    from datasets import load_from_disk

    return [{**row, 'split': split} for split, ds in load_from_disk(str(processed_dir)).items() for row in ds]


def passage_id(context):
    """Mã tài liệu của một đoạn ViQuAD: 16 ký tự hex đầu của SHA-256 nội dung đoạn."""
    return hashlib.sha256(context.encode('utf-8')).hexdigest()[:16]


def viquad_corpus(rows):
    """Trả về {doc_id: {'title', 'text'}} gồm các context duy nhất của mọi split (kể cả test)."""
    corpus = {}
    for row in rows:
        corpus.setdefault(passage_id(row['context']), {'title': row['title'], 'text': row['context']})
    return corpus


def locate_answers(context, answers):
    """Trả về [{'text', 'start', 'end'}] sao cho context[start:end] == text.

    answer_start lệch (ví dụ do bước làm sạch gộp khoảng trắng) thì lấy lần xuất hiện gần nhất;
    đáp án không còn nằm trong đoạn thì bỏ.
    """
    if not answers:
        return []
    pairs = (zip(answers['text'], answers['answer_start']) if isinstance(answers, dict)
             else ((a['text'], a['answer_start']) for a in answers))
    spans = set()
    for text, start in pairs:
        if not text:
            continue
        if context[start:start + len(text)] != text:
            found = [m.start() for m in re.finditer(re.escape(text), context)]
            if not found:
                continue
            start = min(found, key=lambda pos: abs(pos - start))
        spans.add((start, start + len(text)))
    return [{'text': context[s:e], 'start': s, 'end': e} for s, e in sorted(spans)]


def viquad_questions(rows):
    """Câu hỏi dùng được từ train và validation; qid kèm tên split vì id lặp lại giữa các split.

    Câu trả lời được mà không định vị được đáp án nào trong đoạn thì bị bỏ.
    """
    out = []
    for row in rows:
        if row['split'] not in ('train', 'validation'):
            continue
        answers = [] if row['is_impossible'] else locate_answers(row['context'], row['answers'])
        if not row['is_impossible'] and not answers:
            continue
        out.append({'qid': f"{row['split']}:{row['id']}", 'question': row['question'],
                    'answerable': not row['is_impossible'], 'gold_doc_id': passage_id(row['context']),
                    'title': row['title'], 'answers': answers})
    return sorted(out, key=lambda q: q['qid'])


def split_viquad(questions, sizes=SIZES, seed=0):
    """Tách theo title: dev lấy trọn các title ngẫu nhiên tới khi đủ câu; test lấy từ các title còn lại.

    Mỗi đoạn tối đa một câu (cả dev lẫn test). Trả về {tên phần: [câu hỏi]}.
    """
    rng = random.Random(seed)
    questions = list(questions)
    rng.shuffle(questions)
    titles = sorted({q['title'] for q in questions})
    rng.shuffle(titles)

    parts = {name: [] for name in sizes}
    dev_titles, used = set(), set()
    for title in titles:
        if len(parts['dev']) >= sizes['dev']:
            break
        dev_titles.add(title)
        for q in questions:
            if q['title'] == title and q['gold_doc_id'] not in used and len(parts['dev']) < sizes['dev']:
                used.add(q['gold_doc_id'])
                parts['dev'].append(q)

    # Đoạn nào thuộc title của dev thì không vào test, kể cả khi nó lặp lại dưới title khác.
    used |= {q['gold_doc_id'] for q in questions if q['title'] in dev_titles}
    for q in questions:
        name = 'answerable' if q['answerable'] else 'unanswerable'
        if q['title'] not in dev_titles and q['gold_doc_id'] not in used and len(parts[name]) < sizes[name]:
            used.add(q['gold_doc_id'])
            parts[name].append(q)
    return parts


def build_viquad_eval(rows, out_dir, sizes=SIZES, seed=0):
    """Ghi test.jsonl, dev.jsonl, corpus.jsonl, manifest.json; từ chối ghi đè như build_zalo_eval."""
    out = Path(out_dir)
    if out.exists() and any(out.iterdir()):
        raise FileExistsError(f'{out} đã có dữ liệu. Tập đánh giá đã đóng băng; hãy chọn thư mục khác.')

    questions = viquad_questions(rows)
    parts = split_viquad(questions, sizes, seed)
    for name, target in sizes.items():
        if len(parts[name]) != target:
            raise ValueError(f'Phần {name} chỉ có {len(parts[name])}/{target} câu; dữ liệu không đủ.')

    corpus = viquad_corpus(rows)
    out.mkdir(parents=True, exist_ok=True)
    files = {'test.jsonl': parts['answerable'] + parts['unanswerable'], 'dev.jsonl': parts['dev'],
             'corpus.jsonl': [{'doc_id': d, **corpus[d]} for d in sorted(corpus)]}
    for name, data in files.items():
        (out / name).write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in data), encoding='utf-8')

    manifest = {
        'dataset': 'viquad', 'seed': seed, 'target_sizes': sizes,
        'actual_sizes': {name: len(qs) for name, qs in parts.items()},
        'dev_answerable': sum(q['answerable'] for q in parts['dev']),
        'titles': {name: len({q['title'] for q in qs}) for name, qs in parts.items()},
        'usable_questions': len(questions),
        'labeled_rows': sum(row['split'] in ('train', 'validation') for row in rows),
        'corpus_passages': len(corpus),
        'sha256': {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(out.iterdir())},
    }
    (out / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument('dataset', choices=['zalo', 'viquad'])
    parser.add_argument('processed_dir')
    parser.add_argument('out_dir')
    parser.add_argument('--seed', type=int, default=0)
    args = parser.parse_args()
    if args.dataset == 'zalo':
        result = build_zalo_eval(*load_zalo_queries(args.processed_dir), args.out_dir, seed=args.seed)
    else:
        result = build_viquad_eval(load_viquad(args.processed_dir), args.out_dir, seed=args.seed)
    print(json.dumps(result, ensure_ascii=False, indent=2))


Writing /content/src/ragvi/data.py


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/metrics.py`

Runtime Colab không thấy `src/` ở máy cục bộ, nên mã được chép nguyên văn vào `/content/src/`. Mong đợi `Writing /content/src/ragvi/...`.

In [7]:
%%writefile /content/src/ragvi/metrics.py
"""Chỉ số truy xuất: Recall@5, Recall@20, MRR@10, nDCG@5 (tiêu chí chunk đúng theo quyết định D2)."""
import math

METRIC_KEYS = ('r5', 'r20', 'mrr10', 'ndcg5')


def metrics(hits, n_rel):
    """hits: danh sách True/False theo thứ hạng, mỗi tài liệu vàng chỉ True một lần."""
    dcg = sum(h / math.log2(i + 2) for i, h in enumerate(hits[:5]))
    idcg = sum(1 / math.log2(i + 2) for i in range(min(n_rel, 5)))
    return {'r5': sum(hits[:5]) / n_rel, 'r20': sum(hits[:20]) / n_rel,
            'mrr10': next((1 / (i + 1) for i, h in enumerate(hits[:10]) if h), 0.0),
            'ndcg5': dcg / idcg}


def zalo_hits(ranked_doc_ids, gold):
    """Mỗi điều luật vàng chỉ tính một lần, ở lần xuất hiện đầu tiên trong danh sách xếp hạng."""
    seen, hits = set(), []
    for d in ranked_doc_ids:
        hits.append(d in gold and d not in seen)
        seen.add(d)
    return hits


def mean_metrics(rows):
    """Trung bình từng chỉ số trên nhiều truy vấn; rows là các dict có đủ khóa METRIC_KEYS."""
    rows = list(rows)
    if not rows:
        raise ValueError('Không có truy vấn nào để lấy trung bình.')
    return {k: sum(r[k] for r in rows) / len(rows) for k in METRIC_KEYS}


Writing /content/src/ragvi/metrics.py


### [CHẠY MỘT LẦN / MỖI RUNTIME MỚI] Ghi mã `ragvi/retrieval.py`

Runtime Colab không thấy `src/` ở máy cục bộ, nên mã được chép nguyên văn vào `/content/src/`. Mong đợi `Writing /content/src/ragvi/...`.

In [8]:
%%writefile /content/src/ragvi/retrieval.py
"""Truy xuất BM25, dense và hybrid trên Zalo Legal ở mức điều luật (ngày 2–3 của kế hoạch 30 ngày).

Chạy trên Colab, từ thư mục gốc của kho mã:
    PYTHONPATH=src python -m ragvi.retrieval <processed>/zalo_legal <thư mục đầu ra> --methods bm25,dense,hybrid
"""
import argparse
import json
import random
import re
import time
from pathlib import Path

import numpy as np

from ragvi.metrics import METRIC_KEYS, mean_metrics, metrics, zalo_hits

DEFAULT_MODEL = 'BAAI/bge-m3'
# alpha nhân với dense, min–max theo từng truy vấn trên cả kho: chốt 14/11 theo Nguyen & Quan
# (Findings of EACL 2026) và mã ViRE của họ; xem Nhật ký thay đổi trong reports/quyet-dinh-thiet-ke.md.
ALPHA = 0.7
TOP_K = 20


# ---------- BM25 ----------

def tokenize_syllables(text):
    """Tách theo âm tiết: chữ thường, cắt ở khoảng trắng và dấu câu."""
    return re.findall(r'\w+', text.lower())


def build_bm25(texts, tokenize=tokenize_syllables):
    """Dựng chỉ mục BM25Okapi; đổi `tokenize` thành underthesea.word_tokenize để tách từ."""
    from rank_bm25 import BM25Okapi

    return BM25Okapi([tokenize(t) for t in texts])


def bm25_scores(index, query, tokenize=tokenize_syllables):
    """Vector điểm BM25 của một truy vấn trên cả kho; `tokenize` phải trùng với lúc dựng chỉ mục."""
    return index.get_scores(tokenize(query))


def time_word_tokenize(texts, sample=500, seed=0, limit_minutes=30):
    """Đo underthesea.word_tokenize trên `sample` tài liệu ngẫu nhiên, ước lượng cho cả kho.

    Quy tắc của kế hoạch: dưới 30 phút thì E0 dùng underthesea, ngược lại dùng âm tiết.
    """
    from underthesea import word_tokenize

    picked = random.Random(seed).sample(list(texts), min(sample, len(texts)))
    start = time.perf_counter()
    for text in picked:
        word_tokenize(text)
    seconds = time.perf_counter() - start
    # Ước lượng theo số ký tự, vì độ dài điều luật lệch rất mạnh (có điều dài ~250 nghìn ký tự).
    minutes = seconds / max(sum(map(len, picked)), 1) * sum(map(len, texts)) / 60
    return {'sample_docs': len(picked), 'sample_seconds': seconds, 'docs_per_second': len(picked) / seconds,
            'estimated_minutes': minutes, 'use_underthesea': minutes < limit_minutes}


# ---------- Dense ----------

def embedding_cache_path(cache_dir, dataset, chunking, model_name=DEFAULT_MODEL):
    """Đường dẫn file đệm theo khóa (bộ dữ liệu, cách phân đoạn, mô hình)."""
    return Path(cache_dir) / f'{dataset}__{chunking}__{model_name.replace("/", "_")}.npy'


def embed(texts, cache_path=None, model_name=DEFAULT_MODEL, batch_size=32):
    """Nhúng danh sách văn bản thành ma trận float32 đã chuẩn hóa L2; đọc/ghi đệm .npy nếu có `cache_path`."""
    if cache_path is not None and Path(cache_path).exists():
        cached = np.load(cache_path)
        if len(cached) != len(texts):
            raise ValueError(f'File đệm {cache_path} có {len(cached)} dòng nhưng kho có {len(texts)} văn bản; '
                             'hãy xóa file đệm hoặc đổi khóa.')
        return cached

    from sentence_transformers import SentenceTransformer

    model = SentenceTransformer(model_name)
    model.max_seq_length = 512
    start = time.perf_counter()
    vectors = model.encode(list(texts), batch_size=batch_size, normalize_embeddings=True,
                           convert_to_numpy=True, show_progress_bar=True).astype(np.float32)
    seconds = time.perf_counter() - start
    print(f'Đã nhúng {len(texts)} văn bản bằng {model_name} trong {seconds:.1f} giây '
          f'({len(texts) / seconds:.1f} tài liệu/giây).')
    if cache_path is not None:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        np.save(cache_path, vectors)
    return vectors


# ---------- Hybrid và xếp hạng ----------

def minmax(scores):
    """Chuẩn hóa min–max về [0, 1]; vector hằng thì trả về toàn 0."""
    scores = np.asarray(scores, dtype=np.float64)
    span = scores.max() - scores.min()
    return (scores - scores.min()) / span if span > 0 else np.zeros_like(scores)


def hybrid_scores(dense, bm25, alpha=ALPHA):
    """alpha * dense + (1 - alpha) * bm25, mỗi vector điểm được chuẩn hóa min–max trên cả kho."""
    return alpha * minmax(dense) + (1 - alpha) * minmax(bm25)


def top_k(scores, k=TOP_K):
    """Chỉ số của k điểm cao nhất, giảm dần; điểm bằng nhau thì giữ thứ tự trong kho."""
    return np.argsort(-np.asarray(scores), kind='stable')[:k]


# ---------- Bảng kết quả JSONL ----------

def read_rows(path):
    """Đọc các dòng JSONL đã có; bỏ qua dòng cuối bị cắt dở do lượt chạy trước bị ngắt."""
    rows = []
    if Path(path).exists():
        for line in Path(path).read_text(encoding='utf-8').splitlines():
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows


def run_retrieval(scorers, questions, gold, doc_ids, out_path, k=TOP_K):
    """Ghi nối tiếp một dòng cho mỗi cặp (phương pháp, câu hỏi); cặp đã có trong file thì bỏ qua.

    scorers: {tên phương pháp: hàm qid -> vector điểm trên cả kho}. Trả về số dòng mới ghi.
    """
    out = Path(out_path)
    done = {(r['method'], r['qid']) for r in read_rows(out)}
    out.parent.mkdir(parents=True, exist_ok=True)
    dangling = out.exists() and out.stat().st_size > 0 and not out.read_bytes().endswith(b'\n')
    written = 0
    with out.open('a', encoding='utf-8') as f:
        if dangling:  # dòng cuối bị cắt dở: xuống dòng để dòng mới không dính vào
            f.write('\n')
        for method, score in scorers.items():
            for qid in questions:
                if (method, qid) in done:
                    continue
                top = [doc_ids[i] for i in top_k(score(qid), k)]
                row = {'method': method, 'qid': qid, **metrics(zalo_hits(top, gold[qid]), len(gold[qid])),
                       'top': top}
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
                f.flush()
                written += 1
    return written


def summarize(out_path, methods):
    """Trả về {phương pháp: {'n': số câu, r5, r20, mrr10, ndcg5}} từ file JSONL."""
    rows = read_rows(out_path)
    table = {}
    for method in methods:
        own = [r for r in rows if r['method'] == method]
        if own:
            table[method] = {'n': len(own), **mean_metrics(own)}
    return table


def format_table(table):
    header = f'{"Phương pháp":<16}{"n":>6}' + ''.join(f'{name:>11}' for name in
                                                      ('Recall@5', 'Recall@20', 'MRR@10', 'nDCG@5'))
    lines = [f'{method:<16}{row["n"]:>6}' + ''.join(f'{row[k]:>11.4f}' for k in METRIC_KEYS)
             for method, row in table.items()]
    return '\n'.join([header, *lines])


# ---------- CLI ----------

def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument('processed_dir')
    parser.add_argument('out_dir')
    parser.add_argument('--methods', default='bm25,dense,hybrid')
    parser.add_argument('--alpha', type=float, default=ALPHA)
    parser.add_argument('--model', default=DEFAULT_MODEL)
    parser.add_argument('--batch-size', type=int, default=32)
    parser.add_argument('--time-tokenizer', action='store_true',
                        help='đo tốc độ underthesea.word_tokenize trên 500 điều luật rồi thoát')
    args = parser.parse_args(argv)

    from ragvi.data import load_zalo_corpus, load_zalo_queries

    corpus = load_zalo_corpus(args.processed_dir)
    doc_ids, texts = list(corpus), list(corpus.values())
    if args.time_tokenizer:
        print(json.dumps(time_word_tokenize(texts), ensure_ascii=False, indent=2))
        return

    methods = args.methods.split(',')
    unknown = set(methods) - {'bm25', 'dense', 'hybrid'}
    if unknown:
        raise SystemExit(f'Phương pháp không hợp lệ: {sorted(unknown)}')
    questions, gold = load_zalo_queries(args.processed_dir)
    out = Path(args.out_dir)
    out_path = out / 'retrieval_zalo.jsonl'

    # Tham số khác mặc định thì đổi nhãn, để dòng cũ trong JSONL không bị coi là đã xong.
    model_tag = '' if args.model == DEFAULT_MODEL else f'[{args.model}]'
    labels = {'bm25': 'bm25', 'dense': 'dense' + model_tag,
              'hybrid': 'hybrid' + model_tag + ('' if args.alpha == ALPHA else f'[alpha={args.alpha:g}]')}
    done = {(r['method'], r['qid']) for r in read_rows(out_path)}
    pending = [m for m in methods if any((labels[m], q) not in done for q in questions)]

    if {'bm25', 'hybrid'} & set(pending):
        index = build_bm25(texts)
    if {'dense', 'hybrid'} & set(pending):
        cache = out / 'embeddings'
        doc_vectors = embed(texts, embedding_cache_path(cache, 'zalo_legal', 'article', args.model),
                            args.model, args.batch_size)
        query_vectors = embed(list(questions.values()),
                              embedding_cache_path(cache, 'zalo_legal', 'queries', args.model),
                              args.model, args.batch_size)
        row_of = {qid: i for i, qid in enumerate(questions)}

    def bm25(qid):
        return bm25_scores(index, questions[qid])

    def dense(qid):
        return doc_vectors @ query_vectors[row_of[qid]]

    def hybrid(qid):
        return hybrid_scores(dense(qid), bm25(qid), args.alpha)

    scorers = {'bm25': bm25, 'dense': dense, 'hybrid': hybrid}
    written = run_retrieval({labels[m]: scorers[m] for m in pending}, questions, gold, doc_ids, out_path)
    print(f'{len(doc_ids)} tài liệu, {len(questions)} truy vấn; ghi thêm {written} dòng vào {out_path}')
    print(format_table(summarize(out_path, [labels[m] for m in methods])))


if __name__ == '__main__':
    main()


Writing /content/src/ragvi/retrieval.py


### [TÁI SỬ DỤNG] Hàm chạy CLI và ghi log

Đặt `PYTHONPATH` và định nghĩa `run()`: chạy `python -m ragvi....`, in output trực tiếp và lưu vào `O/logs/<tên>.log` để cell cuối tổng hợp. Lệnh lỗi thì dừng cell.

In [9]:
import os
import subprocess
import sys

os.environ['PYTHONPATH'] = '/content/src'
if '/content/src' not in sys.path:
    sys.path.insert(0, '/content/src')
LOG = O / 'logs'
LOG.mkdir(parents=True, exist_ok=True)


def run(name, module, *args):
    cmd = [sys.executable, '-m', module, *map(str, args)]
    print('$', ' '.join(cmd))
    with open(LOG / f'{name}.log', 'w', encoding='utf-8') as log, subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
        for line in proc.stdout:
            print(line, end='')
            log.write(line)
    if proc.returncode:
        raise RuntimeError(f'Lệnh {name} lỗi (mã thoát {proc.returncode}); xem {LOG / name}.log')

### [TÁI SỬ DỤNG] Kiểm tra đọc được dữ liệu đã làm sạch

Đọc `P/zalo_legal/{corpus,queries,qrels}` và `P/viquad` bằng `load_from_disk`. Mong đợi 61.425 điều luật, 818 dòng truy vấn / 788 mã duy nhất; điều kiện tiên quyết cho ngày 13/11.

In [10]:
from datasets import load_from_disk

zalo = {name: load_from_disk(str(Z / name)) for name in ('corpus', 'queries', 'qrels')}
viquad = load_from_disk(str(P / 'viquad'))
checks = [('điều luật (corpus)', len(zalo['corpus']), 61425),
          ('dòng truy vấn', len(zalo['queries']), 818),
          ('mã truy vấn duy nhất', len(set(zalo['queries']['id'])), 788)]
for label, got, want in checks:
    print(f'{label:<22}{got:>8}  mong đợi {want:>6}  {"khớp" if got == want else "KHÔNG KHỚP"}')
print('qrels:', len(zalo['qrels']), 'dòng')
print('ViQuAD:', {split: len(ds) for split, ds in viquad.items()})

điều luật (corpus)       61425  mong đợi  61425  khớp
dòng truy vấn              818  mong đợi    818  khớp
mã truy vấn duy nhất       788  mong đợi    788  khớp
qrels: 793 dòng
ViQuAD: {'train': 28454, 'validation': 3814, 'test': 7301}


### [CHẠY MỘT LẦN / KHI CẬP NHẬT] BM25 trên 788 truy vấn Zalo

Mục tiêu ngày 13/11: bốn chỉ số Recall@5, Recall@20, MRR@10, nDCG@5 của BM25 âm tiết ở mức điều luật. Kết quả nối tiếp vào `O/retrieval/retrieval_zalo.jsonl`; chạy lại chỉ bổ sung truy vấn còn thiếu.

In [11]:
run('retrieval_bm25', 'ragvi.retrieval', Z, O / 'retrieval', '--methods', 'bm25')

$ /usr/bin/python3 -m ragvi.retrieval /content/drive/MyDrive/Vietnamese-RAG-Research/data/processed/zalo_legal /content/drive/MyDrive/Vietnamese-RAG-Research/results/tuan1/retrieval --methods bm25
61425 tài liệu, 788 truy vấn; ghi thêm 788 dòng vào /content/drive/MyDrive/Vietnamese-RAG-Research/results/tuan1/retrieval/retrieval_zalo.jsonl
Phương pháp          n   Recall@5  Recall@20     MRR@10     nDCG@5
bm25               788     0.7005     0.8433     0.5160     0.5529


### [CHẠY MỘT LẦN / KHI CẬP NHẬT] Đo tốc độ underthesea

Mục tiêu ngày 13/11: chọn cách tách từ cho BM25. Mong đợi JSON có `estimated_minutes` và `use_underthesea` (đúng nếu tách cả kho dưới 30 phút).

In [12]:
run('time_tokenizer', 'ragvi.retrieval', Z, O / 'retrieval', '--time-tokenizer')

$ /usr/bin/python3 -m ragvi.retrieval /content/drive/MyDrive/Vietnamese-RAG-Research/data/processed/zalo_legal /content/drive/MyDrive/Vietnamese-RAG-Research/results/tuan1/retrieval --time-tokenizer
{
  "sample_docs": 500,
  "sample_seconds": 9.016481509999949,
  "docs_per_second": 55.45400380907595,
  "estimated_minutes": 19.177208085624027,
  "use_underthesea": true
}


### [CHẠY MỘT LẦN / KHI CẬP NHẬT] Dựng tập đánh giá zalo

Mục tiêu ngày 16/11 (280 trả lời được + 120 không trả lời được, dev 100, kho thu hẹp), seed 0, kèm SHA-256 trong manifest. Tập đã đóng băng: nếu `EVAL/zalo` đã có dữ liệu thì chỉ in manifest hiện có, không dựng lại.

In [13]:
out = EVAL / 'zalo'
if out.exists() and any(out.iterdir()):
    print(f'{out} đã đóng băng, không dựng lại. Manifest hiện có:')
    print((out / 'manifest.json').read_text(encoding='utf-8'))
else:
    run('eval_zalo', 'ragvi.data', 'zalo', Z, out, '--seed', '0')

$ /usr/bin/python3 -m ragvi.data zalo /content/drive/MyDrive/Vietnamese-RAG-Research/data/processed/zalo_legal /content/drive/MyDrive/Vietnamese-RAG-Research/data/eval/zalo --seed 0
{
  "dataset": "zalo_legal",
  "seed": 0,
  "target_sizes": {
    "unanswerable": 120,
    "dev": 100,
    "answerable": 280
  },
  "actual_sizes": {
    "unanswerable": 120,
    "dev": 100,
    "answerable": 280,
    "reserve": 288
  },
  "labeled_queries": 788,
  "groups": 442,
  "removed_docs": 89,
  "sha256": {
    "dev.jsonl": "77f6df10c852d5004ff4ed4d294c9d0b7e004512244b8e90966fb764cc9756bf",
    "removed_doc_ids.json": "c4ff6c30df14220c7d52be3a80699a2d04a93836f10b4dd233f6a0ff95068673",
    "reserve.jsonl": "644e2a51b45982b85d1f30513632b76a7d989416a81c6d2749f41a8639781ce7",
    "test.jsonl": "f0d241c4b1977eab8dcde99cb4260aed9974de85e79aa5f08ee2b3a50885202f"
  }
}


### [CHẠY MỘT LẦN / KHI CẬP NHẬT] Dựng tập đánh giá viquad

Mục tiêu ngày 17/11 (test và dev ViQuAD), seed 0, kèm SHA-256 trong manifest. Tập đã đóng băng: nếu `EVAL/viquad` đã có dữ liệu thì chỉ in manifest hiện có, không dựng lại.

In [14]:
out = EVAL / 'viquad'
if out.exists() and any(out.iterdir()):
    print(f'{out} đã đóng băng, không dựng lại. Manifest hiện có:')
    print((out / 'manifest.json').read_text(encoding='utf-8'))
else:
    run('eval_viquad', 'ragvi.data', 'viquad', P / 'viquad', out, '--seed', '0')

$ /usr/bin/python3 -m ragvi.data viquad /content/drive/MyDrive/Vietnamese-RAG-Research/data/processed/viquad /content/drive/MyDrive/Vietnamese-RAG-Research/data/eval/viquad --seed 0
{
  "dataset": "viquad",
  "seed": 0,
  "target_sizes": {
    "unanswerable": 120,
    "dev": 100,
    "answerable": 280
  },
  "actual_sizes": {
    "unanswerable": 120,
    "dev": 100,
    "answerable": 280
  },
  "dev_answerable": 77,
  "titles": {
    "unanswerable": 74,
    "dev": 3,
    "answerable": 103
  },
  "usable_questions": 32268,
  "labeled_rows": 32268,
  "corpus_passages": 5311,
  "sha256": {
    "corpus.jsonl": "3105f5ad3445497dd5eca4b535f441e41e5798f3c5f88d32de81cead2b447502",
    "dev.jsonl": "578c11353f19c0d8c289df7f3e31f0bcdffd447d0132e47149c8fd081d9075b8",
    "test.jsonl": "60d10838e3ab76996eca9cc21c8882914d5fc505aac611e2f04fc064cd15b14a"
  }
}


### [CHẠY MỘT LẦN / KHI CẬP NHẬT] Thống kê phân đoạn Zalo

Mục tiêu ngày 15/11 và 18/11: số chunk và phân phối độ dài (token bge-m3) của c1, c3, c4, c5, c5_nohead, cùng tỷ lệ điều luật có ít nhất một khoản. Chạy được trên CPU nhưng có thể lâu.

In [15]:
run('chunking', 'ragvi.chunking', Z, '--methods', 'c1,c3,c4,c5,c5_nohead')

$ /usr/bin/python3 -m ragvi.chunking /content/drive/MyDrive/Vietnamese-RAG-Research/data/processed/zalo_legal --methods c1,c3,c4,c5,c5_nohead
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (9231 > 8192). Running this sequence through the model will result in indexing errors
Số tài liệu: 61425
cách        số chunk       tb  trung vị     p95     max
c1            113962    185.0     231.0     256     258
c3            128733    163.7     179.0     256     257
c4             61425    343.1     223.0     946   67395
c5            326759     91.7      77.0     222     717
c5_nohead     326759     64.5      47.0     188     257
Tỷ lệ điều luật có ít nhất một khoản: 84.7%


### [CHẠY MỘT LẦN / KHI CẬP NHẬT] Dense (bge-m3) và hybrid — CẦN GPU

Mục tiêu ngày 14/11: bảng ba dòng BM25, dense, hybrid (α = 0,7). Cần runtime GPU; vector nhúng được đệm ở `O/retrieval/embeddings` nên chạy lại không nhúng lại. Mong đợi in số giây nhúng và bảng chỉ số.

In [16]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Runtime không có GPU. Đổi sang runtime GPU (Runtime > Change runtime type), '
                       'rồi chạy lại từ đầu: cài thư viện, mount Drive, đường dẫn, ghi mã, hàm run.')
run('retrieval_dense_hybrid', 'ragvi.retrieval', Z, O / 'retrieval', '--methods', 'bm25,dense,hybrid')

$ /usr/bin/python3 -m ragvi.retrieval /content/drive/MyDrive/Vietnamese-RAG-Research/data/processed/zalo_legal /content/drive/MyDrive/Vietnamese-RAG-Research/results/tuan1/retrieval --methods bm25,dense,hybrid

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 15855.72it/s]

Batches: 100%|██████████| 1920/1920 [54:47<00:00,  1.71s/it]
Đã nhúng 61425 văn bản bằng BAAI/bge-m3 trong 3288.8 giây (18.7 tài liệu/giây).

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 12731.92it/s]

Batches: 100%|██████████| 25/25 [00:04<00:00,  5.74it/s]
Đã nhúng 788 văn bản bằng BAAI/bge-m3 trong 4.4 giây (180.2 tài liệu/giây).
61425 tài liệu, 788 truy vấn; ghi thêm 1576 dòng vào /content/drive/MyDrive/Vietnamese-RAG-Research/results/tuan1/retrieval/retrieval_zalo.jsonl
Phương pháp          n   Recall@5  Recall@20     MRR@10     nDCG@5
bm25               788     0.7005     0.8433     0.5160     0.5529
dense              788     0.8147     0.9226     0.6525     0.6847
hybrid             788    

### [TÁI SỬ DỤNG] Tổng hợp kết quả cho báo cáo

In gọn bảng chỉ số truy xuất, kết quả đo underthesea, manifest hai tập đánh giá và bảng chunk, để chép vào nhật ký Checklist. Phần nào chưa chạy sẽ ghi "(chưa chạy)".

In [17]:
from ragvi.retrieval import format_table, summarize

print('== Truy xuất Zalo, mức điều luật ==')
print(format_table(summarize(O / 'retrieval' / 'retrieval_zalo.jsonl', ['bm25', 'dense', 'hybrid'])))
for title, path in [('Đo tốc độ underthesea', LOG / 'time_tokenizer.log'),
                    ('Manifest tập đánh giá Zalo', EVAL / 'zalo' / 'manifest.json'),
                    ('Manifest tập đánh giá ViQuAD', EVAL / 'viquad' / 'manifest.json'),
                    ('Phân đoạn Zalo', LOG / 'chunking.log')]:
    print(f'\n== {title} ==')
    print(path.read_text(encoding='utf-8') if path.exists() else '(chưa chạy)')

== Truy xuất Zalo, mức điều luật ==
Phương pháp          n   Recall@5  Recall@20     MRR@10     nDCG@5
bm25               788     0.7005     0.8433     0.5160     0.5529
dense              788     0.8147     0.9226     0.6525     0.6847
hybrid             788     0.8763     0.9581     0.6986     0.7376

== Đo tốc độ underthesea ==
{
  "sample_docs": 500,
  "sample_seconds": 9.016481509999949,
  "docs_per_second": 55.45400380907595,
  "estimated_minutes": 19.177208085624027,
  "use_underthesea": true
}


== Manifest tập đánh giá Zalo ==
{
  "dataset": "zalo_legal",
  "seed": 0,
  "target_sizes": {
    "unanswerable": 120,
    "dev": 100,
    "answerable": 280
  },
  "actual_sizes": {
    "unanswerable": 120,
    "dev": 100,
    "answerable": 280,
    "reserve": 288
  },
  "labeled_queries": 788,
  "groups": 442,
  "removed_docs": 89,
  "sha256": {
    "dev.jsonl": "77f6df10c852d5004ff4ed4d294c9d0b7e004512244b8e90966fb764cc9756bf",
    "removed_doc_ids.json": "c4ff6c30df14220c7d52be3a806